# 06 Resultados, memoria y defensa

Consolida artefactos. No inventa resultados.

Este notebook se ejecuta al final.
Solo consolida artefactos ya generados.
No entrena modelos.
No cambia decisiones.
No inventa resultados.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project_utils import *

RANDOM_STATE = 42
crear_carpetas(".")
sns.set_theme(style="whitegrid")

## Lectura segura de artefactos

Si falta un archivo, se avisa.
La celda no crea metricas falsas.

In [2]:
from pathlib import Path

def leer_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print("Falta artefacto:", path)
    return None

def leer_json(path):
    path = Path(path)
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    print("Falta artefacto:", path)
    return None

## Tablas finales

Se muestran las tablas necesarias para memoria.
La comparativa de test es final.
No se usa para elegir modelos.

In [3]:
dataset_info = leer_json("artifacts/metrics/dataset_info.json")
cv_results = leer_csv("artifacts/metrics/cv_model_selection.csv")
test_results = leer_csv("artifacts/metrics/model_test_results.csv")
cost_results = leer_csv("artifacts/metrics/cost_by_model.csv")
strategy_cv = leer_csv("artifacts/metrics/imbalance_strategy_cv.csv")
anomaly_results = leer_csv("artifacts/metrics/anomaly_results.csv")
robustness = leer_csv("artifacts/metrics/robustness_results.csv")
errors = leer_csv("artifacts/metrics/error_analysis.csv")
shift = leer_csv("artifacts/metrics/shift_results.csv")
importance = leer_csv("artifacts/metrics/permutation_importance.csv")
local_example = leer_csv("artifacts/metrics/local_error_example.csv")
selected_model = leer_json("artifacts/metrics/selected_model_by_cv.json")
selected_strategy = leer_json("artifacts/metrics/selected_imbalance_strategy_by_cv.json")

tablas = {
    "dataset_info": dataset_info,
    "selected_model_by_cv": selected_model,
    "selected_imbalance_strategy_by_cv": selected_strategy,
    "cv_results": cv_results,
    "test_results": test_results,
    "cost_results": cost_results,
    "strategy_cv": strategy_cv,
    "anomaly_results": anomaly_results,
    "robustness": robustness,
    "errors": errors,
    "shift_top20": None if shift is None else shift.head(20),
    "importance_top20": None if importance is None else importance.head(20),
    "local_example": local_example,
}

for nombre, tabla in tablas.items():
    print()
    print(nombre)
    if tabla is None:
        print("No disponible")
    else:
        display(tabla)


dataset_info


{'dataset': 'APS Failure at Scania Trucks',
 'uci_id': 421,
 'n_train': 60000,
 'n_test': 16000,
 'n_features': 170,
 'fp_cost': 10,
 'fn_cost': 500}


selected_model_by_cv


{'selected_model': 'hist_gradient_boosting',
 'selection_metric': 'cv_average_precision',
 'selection_value': 0.8810180976180211,
 'note': 'Seleccionado solo con CV en train'}


selected_imbalance_strategy_by_cv


...

,feature,ks_stat,pvalue
0,br_000,0.022702,0.190590
1,bq_000,0.020786,0.251521
2,ab_000,0.015203,0.515160
3,bp_000,0.014494,0.643237
4,cz_000,0.012359,0.102362
5,cb_000,0.011972,0.055539
6,ay_008,0.011840,0.059944
7,bj_000,0.011837,0.059654
8,ct_000,0.011789,0.133745
9,ad_000,0.011769,0.143031



importance_top20


,feature,importance_mean,importance_std
0,aa_000,0.139433,2.518179e-02
1,ao_000,0.015585,1.291882e-03
2,ay_006,0.014519,8.036471e-03
3,az_001,0.011804,4.289924e-03
4,dx_000,0.008227,4.114017e-03
5,ai_000,0.007979,9.529623e-04
6,ag_004,0.005894,6.327003e-03
7,bl_000,0.005870,5.129974e-03
8,ck_000,0.005591,6.490798e-04
9,ay_008,0.005377,3.795280e-03



local_example


,aa_000,ao_000,ay_006,az_001,dx_000,ai_000,ag_004,bl_000,ck_000,ay_008,y_true,y_pred,tipo_error
0,1055714,48131824.0,29340102.0,111650.0,0.0,24374.0,196186.0,268680.0,8332377.6,3813882.0,1,0,falso_negativo


## Tablas exportadas

Estas tablas se guardan para la memoria.
Solo se guardan si existen los artefactos previos.

In [4]:
if cv_results is not None:
    cv_results.to_csv("artifacts/tables/tabla_cv_memoria.csv", index=False)
if test_results is not None:
    test_results.to_csv("artifacts/tables/tabla_test_memoria.csv", index=False)
if cost_results is not None:
    cost_results.to_csv("artifacts/tables/tabla_costes_memoria.csv", index=False)
if strategy_cv is not None:
    strategy_cv.to_csv("artifacts/tables/tabla_estrategias_cv_memoria.csv", index=False)
if anomaly_results is not None:
    anomaly_results.to_csv("artifacts/tables/tabla_anomalias_memoria.csv", index=False)
if robustness is not None:
    robustness.to_csv("artifacts/tables/tabla_robustez_memoria.csv", index=False)
if shift is not None:
    shift.to_csv("artifacts/tables/tabla_shift_memoria.csv", index=False)
if importance is not None:
    importance.to_csv("artifacts/tables/tabla_importancia_memoria.csv", index=False)
if errors is not None:
    errors.to_csv("artifacts/tables/tabla_errores_memoria.csv", index=False)
print("Tablas guardadas si existian los artefactos.")

Tablas guardadas si existian los artefactos.


## Decisiones metodologicas

- Train y test se separan al principio.
- Test queda reservado para la evaluacion final.
- Los hiperparametros se seleccionan con **GridSearchCV** y 5-fold CV en train.
- El modelo principal se selecciona por PR-AUC medio en CV.
- Cada modelo supervisado usa **Pipeline**.
- El desbalanceo se analiza con metricas adecuadas, coste asimetrico y **class_weight**.
- HistGradientBoostingClassifier es una implementacion de Gradient Boosting en sklearn.
- El coste usa FP igual a 10 y FN igual a 500.
- Los analisis posteriores son auditoria final.
- XAI no prueba causalidad.
- Shift no implica deriva temporal.

## Limitaciones

- Variables anonimizadas.
- Interpretacion limitada por falta de significado fisico.
- Muchos valores faltantes.
- Clase positiva muy minoritaria.
- El coste simplifica la realidad operacional.
- Los detectores de anomalias son secundarios.
- No hay deriva temporal demostrable.
- El modelo requiere supervision humana.

## Preguntas de defensa

- Por que accuracy no basta.
- Por que PR-AUC encaja con desbalanceo.
- Por que el falso negativo cuesta mas.
- Como se evita data leakage.
- Por que HistGradientBoostingClassifier es defendible como Gradient Boosting.
- Por que test no se usa para elegir modelo.
- Que aporta el baseline.
- Que diferencia hay entre Random Forest y Boosting.
- Por que SVM necesita escalado.
- Que significa permutation importance.
- Por que XAI no prueba causalidad.
- Por que anomalias no sustituyen al modelo supervisado.

## Artefactos generados

- **artifacts/metrics/dataset_info.json**.
- **artifacts/metrics/cv_model_selection.csv**.
- **artifacts/metrics/model_test_results.csv**.
- **artifacts/metrics/best_params.json**.
- **artifacts/metrics/selected_model_by_cv.json**.
- **artifacts/metrics/classification_report_test.csv**.
- **artifacts/metrics/imbalance_strategy_cv.csv**.
- **artifacts/metrics/selected_imbalance_strategy_by_cv.json**.
- **artifacts/metrics/cost_by_model.csv**.
- **artifacts/metrics/anomaly_results.csv**.
- **artifacts/metrics/robustness_results.csv**.
- **artifacts/metrics/error_analysis.csv**.
- **artifacts/metrics/shift_results.csv**.
- **artifacts/metrics/permutation_importance.csv**.
- **artifacts/models/best_model.joblib**.
- Figuras en **artifacts/figures**.
- Tablas finales en **artifacts/tables**.

## Cierre

El notebook prepara material para la memoria.
La interpretacion final debe hacerse con los resultados generados al ejecutar.
No se deben escribir metricas no calculadas.